In [32]:
from __future__ import annotations

import os 
from dataclasses import dataclass
from typing import List,Tuple

import cv2
import numpy as np



In [33]:
# Preprocessing
@dataclass
class MatchedObject:
    """One moved-object instance from a match file."""
    match_id: int
    box1: Tuple[float, float, float, float]  # frame-1 box (old position)
    box2: Tuple[float, float, float, float]  # frame-2 box (new position) — training target
    obj_type: int

    
    




In [34]:
def parse_matched_annotation(path: str) -> List[MatchedObject]:
    """Read a `*_match.txt` produced by data_ground_truth_labeller.py.

    Format: 2 rows per moved object,
        row 1 -> frame-1 (old) box
        row 2 -> frame-2 (new) box
    Each row: <match_id> <x> <y> <w> <h> <type>
    """
    rows = []
    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            mid = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
            t = int(parts[5])
            rows.append((mid, x, y, w, h, t))

    if len(rows) % 2 != 0:
        raise ValueError(f"The pair annotation does not have a pair - check")

    matches: List[MatchedObject] = []
    for i in range(0, len(rows), 2):
        r1, r2 = rows[i], rows[i + 1]
        if r1[0] != r2[0]:
            raise ValueError(f"Different match id")
        matches.append(
            MatchedObject(
                match_id=r1[0],
                box1=(r1[1], r1[2], r1[3], r1[4]),
                box2=(r2[1], r2[2], r2[3], r2[4]),
                obj_type=r1[5]
            )
        )
    return matches

In [35]:
# Resize the frames
def frame_resizing(image, height_width: Tuple):
    H, W = height_width
    return cv2.resize(image, (W,H),interpolation=cv2.INTER_LINEAR)

In [36]:
def image_difference(frame_1, frame_2, target_size):
    """Returns
        I_diff: HxWx3 array -> Frame 2 - Frame 1
        F1_resized
        F2_resized
        original_hw"""
        
    f1 = cv2.imread(frame_1)
    f2 = cv2.imread(frame_2)
    
    orig_hw = f2.shape[:2]
    
    f1r = frame_resizing(f1, target_size)
    f2r = frame_resizing(f2, target_size)

    # absolute difference
    diff = cv2.absdiff(f2r, f1r)
    return diff, f1r, f2r, orig_hw
    

In [37]:
# Box rescaling

def box_scaler(box,src_hw: Tuple[int, int],dst_hw: Tuple[int, int]):
    sH, sW = src_hw
    dH, dW = dst_hw
    sx, sy = dW / sW, dH / sH
    x, y, w, h = box
    return (x * sx, y * sy, w * sx, h * sy)

In [38]:
def preprocessing_pipeline(
    frame1_path: str,
    frame2_path: str,
    matched_ann_path: str,
    target_size_hw: Tuple[int, int] = (800, 800),
) -> dict:
    """Full preprocessing for a single training sample.

    Returns:
        Dict: diff image, target boxes and per instance classes
    """
    matches = parse_matched_annotation(matched_ann_path)
    diff, f1r, f2r, orig_hw = image_difference(
        frame1_path, frame2_path, target_size_hw
    )

    target_boxes = [
        box_scaler(m.box2, orig_hw, target_size_hw) for m in matches
    ]
    target_labels = [m.obj_type for m in matches]

    return {
        "diff": diff,                       # HxWx3 uint8, BGR
        "frame1_resized": f1r,
        "frame2_resized": f2r,
        "target_boxes_xywh": target_boxes,  # in resized coords
        "target_labels": target_labels,
        "orig_hw": orig_hw,
        "size_hw": target_size_hw,
        "match_ids": [m.match_id for m in matches],
    }


In [39]:
def resolve_pair_paths(match_filename: str, frames_root: str) -> Tuple[str, str]:
    """Recover (frame1_path, frame2_path) from a match file's name.

    Match filenames look like:
        {folder}-{frame1_basename}-{frame2_basename}_match.txt
    where `folder` itself has no '-'. We split on '-' from both ends.
    """
    name = os.path.basename(match_filename)
    if name.endswith("_match.txt"):
        name = name[: -len("_match.txt")]
    parts = name.split("-")
    if len(parts) != 3:
        raise ValueError(f"unexpected match filename: {match_filename}")
    folder, f1, f2 = parts
    p1 = os.path.join(frames_root, folder, f1 + ".png")
    p2 = os.path.join(frames_root, folder, f2 + ".png")
    return p1, p2

In [40]:
MATCH_DIR =  "/Users/alex/Documents/NYU/Classes/CS_3033_Computer_Vision/CS_3033_Vision_Meets_ML/Assignment_4/data/matched_annotations"
FRAMES_ROOT = "/Users/alex/Documents/NYU/Classes/CS_3033_Computer_Vision/CS_3033_Vision_Meets_ML/Assignment_3/data"

sample = sorted(os.listdir(MATCH_DIR))[0]
p1, p2 = resolve_pair_paths(sample, FRAMES_ROOT)
out = preprocessing_pipeline(p1, p2, os.path.join(MATCH_DIR, sample))
print(f"sample          : {sample}")
print(f"frame1          : {p1}")
print(f"frame2          : {p2}")
print(f"orig_hw         : {out['orig_hw']}")
print(f"size_hw         : {out['size_hw']}")
print(f"diff shape/dtype: {out['diff'].shape} {out['diff'].dtype}")
print(f"diff stats      : min={out['diff'].min()} max={out['diff'].max()} "
        f"mean={out['diff'].mean():.2f}")
print(f"# moved objects : {len(out['target_boxes_xywh'])}")
for i, (b, t) in enumerate(zip(out['target_boxes_xywh'], out['target_labels'])):
    print(f"  obj {i}: type={t} box(xywh)={tuple(round(v, 1) for v in b)}")


sample          : Pair_S_000001_3503_3563-S_000001_frame_3503-S_000001_frame_3563_match.txt
frame1          : /Users/alex/Documents/NYU/Classes/CS_3033_Computer_Vision/CS_3033_Vision_Meets_ML/Assignment_3/data/Pair_S_000001_3503_3563/S_000001_frame_3503.png
frame2          : /Users/alex/Documents/NYU/Classes/CS_3033_Computer_Vision/CS_3033_Vision_Meets_ML/Assignment_3/data/Pair_S_000001_3503_3563/S_000001_frame_3563.png
orig_hw         : (1080, 1920)
size_hw         : (800, 800)
diff shape/dtype: (800, 800, 3) uint8
diff stats      : min=0 max=246 mean=3.24
# moved objects : 1
  obj 0: type=1 box(xywh)=(205.0, 443.0, 22.5, 98.5)


## Part 2 — Input to DETR

Feed `I_diff` (the absolute pixel difference) into the **full** pre-trained DETR
model from HuggingFace (`facebook/detr-resnet-50`). The model internally chains:

1. **ResNet-50 backbone** → feature map of shape `C × H/32 × W/32`
2. **Transformer encoder** → globally context-mixed features (with positional encoding)
3. **Transformer decoder** with `N` learned object queries → `N` slot embeddings
4. **Detection heads** → per-query (class logits, bbox in `cxcywh`, normalized to [0,1])

We do **not** reimplement any of that. We load the pretrained weights and swap
only the classification head for our 6 classes
(0=Unknown, 1=Person, 2=Car, 3=Other Vehicle, 4=Other Object, 5=Bike).

In [41]:
import torch
from transformers import DetrImageProcessor, DetrForObjectDetection

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
NUM_CLASSES = 6  # 0=Unknown 1=Person 2=Car 3=OtherVehicle 4=OtherObject 5=Bike
print(f"device      : {DEVICE}")

device      : mps


In [42]:
# Load pretrained DETR + image processor.
# - num_labels=NUM_CLASSES re-initializes the classification head (91 COCO classes -> 6 ours).
#   DETR also reserves an extra "no object" slot internally, so logits shape will be (N, NUM_CLASSES+1).
# - ignore_mismatched_sizes=True allows the head to reset cleanly without crashing on shape mismatch.

MODEL_NAME = "facebook/detr-resnet-50"

processor = DetrImageProcessor.from_pretrained(MODEL_NAME)

model = DetrForObjectDetection.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
).to(DEVICE)

print("num_queries :", model.config.num_queries)   # 100 by default
print("num_labels  :", model.config.num_labels)    # should be 6
print("d_model     :", model.config.d_model)       # 256 (transformer hidden dim)

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `91`.
Loading weights: 100%|██████████| 530/530 [00:00<00:00, 7863.53it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |                                                                                        
---------------------------------------------------------------+------------+----------------------------------------------------------------------------------------
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |         

num_queries : 100
num_labels  : 6
d_model     : 256


In [43]:
def forward_diff(diff_bgr, model, processor):
    """Single-image forward pass.

    diff_bgr : HxWx3 uint8 (OpenCV BGR) from preprocessing_pipeline()['diff']
    The processor handles its own resize + ImageNet normalize internally.
    """
    diff_rgb = cv2.cvtColor(diff_bgr, cv2.COLOR_BGR2RGB)             # cv2 returns BGR, DETR wants RGB
    inputs = processor(images=diff_rgb, return_tensors="pt").to(DEVICE)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs

# Smoke test on the I_diff we built in Part 1
outputs = forward_diff(out["diff"], model, processor)
print("logits     :", tuple(outputs.logits.shape))      # (1, num_queries, NUM_CLASSES+1)
print("pred_boxes :", tuple(outputs.pred_boxes.shape))  # (1, num_queries, 4)  cxcywh, normalized [0,1]
print("box[0,0]   :", outputs.pred_boxes[0, 0].cpu().numpy())

logits     : (1, 100, 7)
pred_boxes : (1, 100, 4)
box[0,0]   : [0.01042336 0.8125524  0.02075137 0.08650792]


## Part 3 — Train/Test Split + Dataset

We need a `torch.utils.data.Dataset` that:

1. Lists every `*_match.txt` in `MATCH_DIR` (one match file = one training sample = one frame pair).
2. Splits the list 80/20 (train/test) with a fixed seed for reproducibility.
3. Per sample: runs `preprocessing_pipeline` to get `I_diff` + frame-2 boxes + class labels.
4. Wraps the boxes into **COCO-style** annotations (`xywh` in pixels + `category_id`),
   which is the format `DetrImageProcessor` expects.
5. A `collate_fn` calls the processor on the batch — the processor returns
   `pixel_values`, `pixel_mask`, and `labels` (DETR's normalized `cxcywh` boxes
   + class IDs) ready to feed straight into the model.

In [44]:
import random

SEED         = 42
TARGET_SIZE  = (800, 800)   # H, W — matches what we feed through DETR
TRAIN_FRAC   = 0.8

# Enumerate match files, drop any whose frame .pngs are missing on disk.
all_matches = sorted(os.listdir(MATCH_DIR))

def _frames_exist(match_filename):
    p1, p2 = resolve_pair_paths(match_filename, FRAMES_ROOT)
    return os.path.isfile(p1) and os.path.isfile(p2)

all_matches = [m for m in all_matches if _frames_exist(m)]

# Deterministic shuffle so train/test split is stable across runs.
rng = random.Random(SEED)
rng.shuffle(all_matches)

n_train     = int(TRAIN_FRAC * len(all_matches))
train_files = all_matches[:n_train]
test_files  = all_matches[n_train:]

print(f"total : {len(all_matches)}")
print(f"train : {len(train_files)} ({len(train_files) / len(all_matches):.0%})")
print(f"test  : {len(test_files)} ({len(test_files) / len(all_matches):.0%})")

total : 290
train : 232 (80%)
test  : 58 (20%)


In [45]:
from torch.utils.data import Dataset, DataLoader


class MovedObjectDataset(Dataset):
    """One sample = one matched frame pair.

    __getitem__ returns a dict {"image": HxWx3 uint8 RGB, "target": COCO-style annotation}.
    Batching + tensor conversion is deferred to `collate_fn` so the HF processor can
    do resize, normalize, and box format conversion in one place.
    """

    def __init__(self, match_files, match_dir, frames_root, target_size_hw=(800, 800)):
        self.match_files = match_files
        self.match_dir = match_dir
        self.frames_root = frames_root
        self.target_size = target_size_hw

    def __len__(self):
        return len(self.match_files)

    def __getitem__(self, idx):
        name = self.match_files[idx]
        p1, p2 = resolve_pair_paths(name, self.frames_root)
        sample = preprocessing_pipeline(
            p1, p2,
            os.path.join(self.match_dir, name),
            target_size_hw=self.target_size,
        )
        diff_rgb = cv2.cvtColor(sample["diff"], cv2.COLOR_BGR2RGB)

        # COCO-style annotations: bbox in xywh (pixels, in resized coords).
        # The processor will convert to DETR's normalized cxcywh internally.
        annotations = []
        for box, label in zip(sample["target_boxes_xywh"], sample["target_labels"]):
            x, y, w, h = box
            annotations.append({
                "bbox": [float(x), float(y), float(w), float(h)],
                "category_id": int(label),
                "area": float(w * h),
                "iscrowd": 0,
            })
        target = {"image_id": idx, "annotations": annotations}
        return {"image": diff_rgb, "target": target}


def collate_fn(batch):
    images  = [b["image"] for b in batch]
    targets = [b["target"] for b in batch]
    enc = processor(images=images, annotations=targets, return_tensors="pt")
    return {
        "pixel_values": enc["pixel_values"],
        "pixel_mask":   enc["pixel_mask"],
        "labels":       enc["labels"],   # list of dicts: class_labels (LongTensor), boxes (cxcywh, normalized)
    }


train_ds = MovedObjectDataset(train_files, MATCH_DIR, FRAMES_ROOT, TARGET_SIZE)
test_ds  = MovedObjectDataset(test_files,  MATCH_DIR, FRAMES_ROOT, TARGET_SIZE)

train_dl = DataLoader(train_ds, batch_size=2, shuffle=True,  collate_fn=collate_fn, num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"train_ds len: {len(train_ds)}")
print(f"test_ds  len: {len(test_ds)}")

train_ds len: 232
test_ds  len: 58


In [46]:
# Smoke test: pull one batch, send it through the model end-to-end.
batch = next(iter(train_dl))
print("pixel_values:", tuple(batch["pixel_values"].shape))   # (B, 3, H, W)
print("pixel_mask  :", tuple(batch["pixel_mask"].shape))     # (B, H, W)
for i, lab in enumerate(batch["labels"]):
    print(f"  sample {i}: class_labels={lab['class_labels'].tolist()}  "
          f"boxes(cxcywh, norm)={lab['boxes'].tolist()}")

# Forward pass — DETR returns loss when labels are provided.
model.train()
out_train = model(
    pixel_values=batch["pixel_values"].to(DEVICE),
    pixel_mask=batch["pixel_mask"].to(DEVICE),
    labels=[{k: v.to(DEVICE) for k, v in lab.items()} for lab in batch["labels"]],
)
print(f"loss        : {out_train.loss.item():.4f}")
print(f"loss_dict   : {{ {', '.join(f'{k}={v.item():.3f}' for k, v in out_train.loss_dict.items())} }}")

pixel_values: (2, 3, 800, 800)
pixel_mask  : (2, 800, 800)
  sample 0: class_labels=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2]  boxes(cxcywh, norm)=[[0.6875, 0.7701388597488403, 0.015625, 0.07916664332151413], [0.602343738079071, 0.8118055462837219, 0.01718750037252903, 0.08472221344709396], [0.6363281011581421, 0.7972221970558167, 0.01796874962747097, 0.09444442391395569], [0.604296863079071, 0.7548611164093018, 0.01640624925494194, 0.07361114770174026], [0.6953125, 0.7409722208976746, 0.02031249925494194, 0.08749999850988388], [0.6187499761581421, 0.7555555701255798, 0.01718750037252903, 0.07222221046686172], [0.6820312738418579, 0.7138888835906982, 0.01875000074505806, 0.07500000298023224], [0.713671863079071, 0.7395833730697632, 0.01796874962747097, 0.08194442838430405], [0.621874988079071, 0.8006944060325623, 0.02187499962747097, 0.09305557608604431], [0.705078125, 0.7284722328186035, 0.01796874962747097, 0.08194442838430405], [0.9203125238418579, 0.7319444417953491, 0.05781250074505806, 0

## Part 4 — Freeze Strategies

The assignment requires comparing **three** fine-tuning strategies on the same DETR
model. Each strategy unfreezes a different *contiguous* block of the network and
freezes everything else:

| #   | Strategy           | Trainable                                                         | Frozen                                              |
|-----|--------------------|-------------------------------------------------------------------|-----------------------------------------------------|
| 4.1 | Backbone only      | ResNet-50 (`model.backbone.*`)                                    | input projection, encoder, decoder, heads           |
| 4.2 | Heads only         | `class_labels_classifier`, `bbox_predictor`                       | backbone, input projection, encoder, decoder, queries |
| 4.3 | Transformer only   | `model.encoder.*`, `model.decoder.*`, `model.query_position_embeddings` | backbone, input projection, heads                   |

`model.input_projection` (the 1×1 conv that maps backbone features 2048 → 256) is
left **frozen** in every strategy — it's neither part of the backbone nor of the
transformer, and the assignment doesn't address it. Document this choice in the report.

The helper below sets `requires_grad` accordingly and returns the list of params
the optimizer should see.

In [47]:
STRATEGY_PREFIXES = {
    "backbone":    ("model.backbone.",),
    "heads":       ("class_labels_classifier.", "bbox_predictor."),
    "transformer": ("model.encoder.", "model.decoder.", "model.query_position_embeddings"),
}


def freeze_for_strategy(model, strategy: str):
    """Freeze the whole model, then re-enable grads on the params matching `strategy`.

    Returns the list of trainable parameters — pass this directly to the optimizer.
    """
    if strategy not in STRATEGY_PREFIXES:
        raise ValueError(f"unknown strategy: {strategy}. choose from {list(STRATEGY_PREFIXES)}")
    prefixes = STRATEGY_PREFIXES[strategy]

    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if any(name.startswith(pfx) for pfx in prefixes):
            p.requires_grad = True

    return [p for p in model.parameters() if p.requires_grad]


def summarize_trainable(model):
    groups = {"backbone": 0, "input_projection": 0, "encoder": 0, "decoder": 0,
              "queries": 0, "class_head": 0, "bbox_head": 0, "other": 0}
    trainable = total = 0
    for name, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
            if   name.startswith("model.backbone."):                  groups["backbone"]         += p.numel()
            elif name.startswith("model.input_projection"):           groups["input_projection"] += p.numel()
            elif name.startswith("model.encoder."):                   groups["encoder"]          += p.numel()
            elif name.startswith("model.decoder."):                   groups["decoder"]          += p.numel()
            elif name.startswith("model.query_position_embeddings"): groups["queries"]          += p.numel()
            elif name.startswith("class_labels_classifier"):          groups["class_head"]       += p.numel()
            elif name.startswith("bbox_predictor"):                   groups["bbox_head"]        += p.numel()
            else:                                                     groups["other"]            += p.numel()
    print(f"  trainable / total : {trainable:,} / {total:,}  ({100 * trainable / total:.2f}%)")
    for g, n in groups.items():
        if n:
            print(f"    {g:<17}: {n:,}")

### 4.1 — Backbone only (ResNet-50)

Update only `model.backbone.*` (the ResNet-50 conv stack). Freeze the input
projection, transformer encoder/decoder, object queries, and both detection heads.

**Why this might work:** the difference image is visually very different from
ImageNet/COCO photos (mostly black with sparse motion blobs). The ResNet's
low-level filters likely need to re-learn what an "edge in a diff image" looks
like before downstream layers can do their job.

**Why this might fail:** the transformer + heads were trained against ResNet
features in COCO's distribution. If we move the backbone but freeze them, we
break the alignment they assumed.

In [48]:
params_backbone = freeze_for_strategy(model, "backbone")
print("strategy: backbone")
summarize_trainable(model)

strategy: backbone
  trainable / total : 23,454,912 / 41,502,923  (56.51%)
    backbone         : 23,454,912


### 4.2 — Detection heads only

Update only the final fully-connected predictors:

- `class_labels_classifier` — `Linear(d_model=256 → num_classes+1=7)`
- `bbox_predictor` — 3-layer MLP that emits `cxcywh` per query.

Freeze the backbone, encoder, decoder, and object queries.

**Why this might work:** smallest parameter count → cheapest to fit; if the
COCO-pretrained backbone + transformer features already separate "static
background" from "moved blob", we just need to remap their semantics to our 6
classes.

**Why this might fail:** the diff image is out-of-distribution for the frozen
feature extractor. If the upstream features don't expose useful "moved object"
information, no linear classifier can recover it.

In [49]:
params_heads = freeze_for_strategy(model, "heads")
print("strategy: heads")
summarize_trainable(model)

strategy: heads
  trainable / total : 134,411 / 41,502,923  (0.32%)
    class_head       : 1,799
    bbox_head        : 132,612


### 4.3 — Transformer encoder + decoder (+ object queries)

Update `model.encoder.*`, `model.decoder.*`, and the learned
`model.query_position_embeddings`. Freeze the backbone and both detection heads.

**Why this might work:** the transformer is what *binds* spatial features to
object queries. If diff images make the "where to attend" problem genuinely
different from natural images, the attention maps need to re-learn what to
focus on, while the backbone's features and the head's class mapping can stay
COCO-aligned (people stay people, vehicles stay vehicles).

**Why this might fail:** transformer is the largest parameter group (~30 M
params) and the most data-hungry. With our small dataset it may overfit, and
freezing the heads constrains what the transformer can output.

In [50]:
params_transformer = freeze_for_strategy(model, "transformer")
print("strategy: transformer")
summarize_trainable(model)

strategy: transformer
  trainable / total : 17,389,056 / 41,502,923  (41.90%)
    encoder          : 7,890,432
    decoder          : 9,473,024
    queries          : 25,600


## Part 5 — Training Loop

Standard PyTorch loop. `train_epoch` runs one pass over `train_dl` and returns
the average loss. The driver below picks a strategy, freezes the model, builds
an `AdamW` optimizer over the trainable params, and calls `train_epoch` once
per epoch.

In [51]:
from torch.optim import AdamW


def train_epoch(model, train_loader, optimizer, device):
    """One pass over train_loader. Returns average loss."""
    model.train()
    total_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask   = batch["pixel_mask"].to(device)
        labels       = [{k: v.to(device) for k, v in lab.items()} for lab in batch["labels"]]

        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 20 == 0:
            print(f"  Batch [{batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}")

    return total_loss / len(train_loader)

### 5.1 — Train one strategy

Freeze, build optimizer, loop. Re-run the model-load cell (Part 2) before
training another strategy so you start from the pretrained weights again.

In [52]:
strategy   = "heads"
num_epochs = 5
lr         = 1e-4

freeze_for_strategy(model, strategy)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

train_losses = []
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    avg_loss = train_epoch(model, train_dl, optimizer, DEVICE)
    train_losses.append(avg_loss)
    print(f"  avg train loss: {avg_loss:.4f}")

Epoch 1/5
  Batch [0/116] Loss: 3.4680
  Batch [20/116] Loss: 3.5014
  Batch [40/116] Loss: 3.1651
  Batch [60/116] Loss: 1.8288
  Batch [80/116] Loss: 1.9397
  Batch [100/116] Loss: 3.1972
  avg train loss: 2.5242
Epoch 2/5
  Batch [0/116] Loss: 3.1918
  Batch [20/116] Loss: 2.3049
  Batch [40/116] Loss: 2.7349


KeyboardInterrupt: 

### 5.2 — Compare all 3 strategies

Same loop, three times. Each strategy starts from a freshly-loaded pretrained
DETR so the comparison is fair. Slow on a laptop — usually run on the cluster.

In [53]:
num_epochs = 10
lrs = {"backbone": 1e-5, "heads": 1e-4, "transformer": 1e-4}

all_losses = {}

for strategy in ["backbone", "heads", "transformer"]:
    print(f"\n=== Training {strategy} ===")

    # Fresh pretrained model for each strategy.
    model = DetrForObjectDetection.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
    ).to(DEVICE)
    freeze_for_strategy(model, strategy)
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lrs[strategy])

    losses = []
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        avg_loss = train_epoch(model, train_dl, optimizer, DEVICE)
        losses.append(avg_loss)
        print(f"  avg train loss: {avg_loss:.4f}")

    all_losses[strategy] = losses
    torch.save(model.state_dict(), f"{strategy}.pt")


=== Training backbone ===


[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `91`.
Loading weights: 100%|██████████| 530/530 [00:00<00:00, 8788.78it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |                                                                                        
---------------------------------------------------------------+------------+----------------------------------------------------------------------------------------
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |         

Epoch 1/10
  Batch [0/116] Loss: 6.1518


KeyboardInterrupt: 

## Part 6 — Evaluation + Visualization

Training optimizes the DETR composite loss (`loss_ce + loss_bbox + loss_giou`),
which is a fine objective but not directly interpretable. For the report we need:

- **Precision** = TP / (TP + FP) — of all predicted boxes, what fraction matched a moved object.
- **Recall**    = TP / (TP + FN) — of all ground-truth moved objects, what fraction the model detected.
- **F1**        = harmonic mean of the two — used as a single "accuracy" number for plots.

A prediction is a **TP** if its IoU with some ground-truth box is `≥ iou_threshold`
(default 0.5). Each GT can be matched to at most one prediction (greedy match,
highest-IoU first).

In [ ]:
def box_iou(b1, b2):
    """IoU between two xyxy boxes (each a length-4 array/list)."""
    x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    return inter / (a1 + a2 - inter + 1e-9)


def cxcywh_norm_to_xyxy(boxes_norm, H, W):
    """Convert (M,4) cxcywh-in-[0,1] to (M,4) xyxy in pixels."""
    out = []
    for cx, cy, w, h in boxes_norm:
        cx, cy, w, h = cx * W, cy * H, w * W, h * H
        out.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
    return np.array(out) if out else np.zeros((0, 4))


@torch.no_grad()
def evaluate(model, test_loader, device, score_threshold=0.5, iou_threshold=0.5):
    """Greedy IoU matching of post-processed predictions vs. ground truth.

    Returns precision, recall, F1 averaged over the whole test set.
    """
    model.eval()
    tp = fp = fn = 0

    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask   = batch["pixel_mask"].to(device)
        labels       = batch["labels"]   # ground truth, list of dicts (cxcywh, normalized)

        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
        target_sizes = torch.tensor([[TARGET_SIZE[0], TARGET_SIZE[1]]] * len(labels))
        preds = processor.post_process_object_detection(
            outputs, threshold=score_threshold, target_sizes=target_sizes
        )

        for pred, gt in zip(preds, labels):
            pred_boxes = pred["boxes"].cpu().numpy()                                          # (N, 4) xyxy in 800x800
            gt_boxes   = cxcywh_norm_to_xyxy(gt["boxes"].cpu().numpy(),
                                             TARGET_SIZE[0], TARGET_SIZE[1])                  # (M, 4) xyxy

            matched = set()
            for pb in pred_boxes:
                best_iou, best_j = 0.0, -1
                for j, gb in enumerate(gt_boxes):
                    if j in matched:
                        continue
                    iou = box_iou(pb, gb)
                    if iou > best_iou:
                        best_iou, best_j = iou, j
                if best_iou >= iou_threshold:
                    tp += 1
                    matched.add(best_j)
                else:
                    fp += 1
            fn += len(gt_boxes) - len(matched)

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

### 6.1 — Train all 3 strategies, track loss + metrics per epoch

Same shape as 5.2, but after each epoch we also call `evaluate` on the test set
and store precision / recall / F1. Everything ends up in `results[strategy]`,
ready for plotting.

In [ ]:
num_epochs = 10
lrs = {"backbone": 1e-5, "heads": 1e-4, "transformer": 1e-4}

results = {}

for strategy in ["backbone", "heads", "transformer"]:
    print(f"\n=== Training {strategy} ===")

    model = DetrForObjectDetection.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
    ).to(DEVICE)
    freeze_for_strategy(model, strategy)
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lrs[strategy])

    losses, precisions, recalls, f1s = [], [], [], []
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        avg_loss = train_epoch(model, train_dl, optimizer, DEVICE)
        m = evaluate(model, test_dl, DEVICE)
        losses.append(avg_loss)
        precisions.append(m["precision"])
        recalls.append(m["recall"])
        f1s.append(m["f1"])
        print(f"  loss={avg_loss:.4f}  P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}")

    results[strategy] = {
        "loss":      losses,
        "precision": precisions,
        "recall":    recalls,
        "f1":        f1s,
    }
    torch.save(model.state_dict(), f"{strategy}.pt")

### 6.2 — Loss & accuracy curves

Two side-by-side plots: train loss (drops over epochs) and F1 score (rises),
one line per strategy.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))

for strategy, r in results.items():
    epochs = range(1, len(r["loss"]) + 1)
    ax_loss.plot(epochs, r["loss"],  marker="o", label=strategy)
    ax_acc.plot (epochs, r["f1"],    marker="o", label=strategy)

ax_loss.set_title("Training loss")
ax_loss.set_xlabel("Epoch"); ax_loss.set_ylabel("Loss")
ax_loss.grid(True); ax_loss.legend()

ax_acc.set_title("F1 score on test set")
ax_acc.set_xlabel("Epoch"); ax_acc.set_ylabel("F1")
ax_acc.set_ylim(0, 1)
ax_acc.grid(True); ax_acc.legend()

plt.tight_layout()
plt.savefig("loss_and_f1.png", dpi=120)
plt.show()

### 6.3 — Qualitative visualizations

Pick 5 random test pairs, run the trained model on each `I_diff`, and overlay
predicted boxes (red) and ground truth (green) on the resized frame 2. The
assignment requires at least 5 examples in the report.

In [ ]:
# Pick which trained model to visualize. After 6.1, `model` holds the last strategy
# (transformer). To visualize a different one, reload its checkpoint:
#   model.load_state_dict(torch.load("heads.pt"))
model.eval()

num_samples = 5
score_threshold = 0.5

rng_vis = np.random.default_rng(0)
chosen = rng_vis.choice(len(test_files), size=num_samples, replace=False)

fig, axes = plt.subplots(num_samples, 2, figsize=(12, 4 * num_samples))

for i, idx in enumerate(chosen):
    name = test_files[idx]
    p1, p2 = resolve_pair_paths(name, FRAMES_ROOT)
    sample = preprocessing_pipeline(p1, p2, os.path.join(MATCH_DIR, name), TARGET_SIZE)
    diff_rgb = cv2.cvtColor(sample["diff"], cv2.COLOR_BGR2RGB)
    f2_rgb   = cv2.cvtColor(sample["frame2_resized"], cv2.COLOR_BGR2RGB)

    inputs = processor(images=diff_rgb, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    target_sizes = torch.tensor([[TARGET_SIZE[0], TARGET_SIZE[1]]])
    pred = processor.post_process_object_detection(
        outputs, threshold=score_threshold, target_sizes=target_sizes
    )[0]

    # Left: I_diff with predictions
    axes[i, 0].imshow(diff_rgb)
    axes[i, 0].set_title(f"I_diff — {name[:40]}…")
    axes[i, 0].axis("off")
    for box, score, label in zip(pred["boxes"], pred["scores"], pred["labels"]):
        x1, y1, x2, y2 = box.cpu().numpy()
        axes[i, 0].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="red", linewidth=2))
        axes[i, 0].text(x1, max(0, y1 - 4), f"{int(label)}:{score:.2f}",
                        color="red", fontsize=8)

    # Right: Frame 2 with GT (green) and predictions (red)
    axes[i, 1].imshow(f2_rgb)
    axes[i, 1].set_title("Frame 2 — green=GT, red=pred")
    axes[i, 1].axis("off")
    for x, y, w, h in sample["target_boxes_xywh"]:
        axes[i, 1].add_patch(plt.Rectangle((x, y), w, h, fill=False,
                                           edgecolor="lime", linewidth=2))
    for box in pred["boxes"]:
        x1, y1, x2, y2 = box.cpu().numpy()
        axes[i, 1].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="red", linewidth=2))

plt.tight_layout()
plt.savefig("qualitative_predictions.png", dpi=120)
plt.show()